# DDRI 대표 15개 오류 스테이션 분석

- 목적: 대표 15개 `station-hour(대여소-시간 단위)` 실험에서 상위 오류 스테이션을 우선 해석하고, 전체 161개 상위 오류와의 겹침 및 핵심 시간대 오차 패턴을 정리한다.
- 범위: 대표 15개 기준 오류 우선순위, 군집별 평균 오류, `2377`/`2348` 시간대별 실제값-예측값 차이
- 입력:
  - `works/05_prediction_long/output/data/ddri_station_hour_station_error_summary.csv`
  - `works/06_prediction_long_full/output/data/ddri_full_top20_error_rep15_overlap(겹침).csv`
  - `3조 공유폴더/대표대여소_예측데이터_15개/raw_data/`
- 생성 산출물:
  - `rep15_error_analysis/output/data/ddri_rep15_station_error_priority_table.csv`
  - `rep15_error_analysis/output/data/ddri_rep15_station_error_cluster_summary.csv`
  - `rep15_error_analysis/output/data/ddri_rep15_top2_station_hourly_error_patterns.csv`
  - `rep15_error_analysis/output/data/ddri_rep15_top2_station_error_summary.csv`
  - `rep15_error_analysis/output/data/ddri_rep15_top2_station_peak_error_hours.csv`


## 용어 설명
- `station-hour(대여소-시간 단위)`: 대여소 하나의 특정 시간대를 한 행으로 보는 분석 단위
- `MAE(평균절대오차)`: 실제 대여량과 예측 대여량 차이의 절대값 평균
- `gap_mean(평균 차이)`: 과대 예측인지 과소 예측인지 방향까지 포함한 평균 차이
- `Top20(상위 20개)`: 오류가 큰 순서대로 뽑은 상위 대여소 목록
- `subset(축소 피처 조합)`: 오류가 큰 군집에서 다시 시험해 보는 피처 묶음


In [1]:

from pathlib import Path

import pandas as pd
from lightgbm import LGBMRegressor

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
ANALYSIS_DIR = ROOT / 'works/05_prediction_long/cheng80/rep15_error_analysis'
OUTPUT_DATA_DIR = ANALYSIS_DIR / 'output/data'
RAW_DATA_DIR = ROOT / '3조 공유폴더/대표대여소_예측데이터_15개/raw_data'
REP_ERR_PATH = ROOT / 'works/05_prediction_long/output/data/ddri_station_hour_station_error_summary.csv'
FULL_OVERLAP_PATH = ROOT / 'works/06_prediction_long_full/output/data/ddri_full_top20_error_rep15_overlap.csv'
TRAIN_PATH = RAW_DATA_DIR / 'ddri_prediction_long_train_2023_2024.csv'
TEST_PATH = RAW_DATA_DIR / 'ddri_prediction_long_test_2025.csv'

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)



## 1. 대표 15개 오류 우선순위 정리

대표 15개 스테이션 오류 요약표와 전체 161개 상위 오류 overlap(겹침) 표를 합쳐서, 지금 먼저 해석할 우선순위를 고정한다.


In [2]:

rep_err = pd.read_csv(REP_ERR_PATH, encoding='utf-8-sig')
rep_err['station_id'] = rep_err['station_id'].astype(str)

rep15 = pd.read_csv(TEST_PATH, encoding='utf-8-sig')[['station_id', 'station_name', 'cluster']].drop_duplicates()
rep15['station_id'] = rep15['station_id'].astype(str)

full_overlap = pd.read_csv(FULL_OVERLAP_PATH, encoding='utf-8-sig')
full_overlap['station_id'] = full_overlap['station_id'].astype(str)
full_top20_ids = set(full_overlap['station_id'])

priority_df = rep_err.merge(rep15[['station_id', 'cluster']], on='station_id', how='left')
priority_df['is_full_top20_overlap'] = priority_df['station_id'].isin(full_top20_ids)
priority_df = priority_df.sort_values(['rmse', 'mae'], ascending=False).reset_index(drop=True)
priority_df.insert(0, 'error_rank', priority_df.index + 1)
priority_df['priority_note'] = ''
priority_df.loc[priority_df['error_rank'] <= 5, 'priority_note'] = '대표 15개 우선 해석 상위권'
priority_df.loc[priority_df['is_full_top20_overlap'], 'priority_note'] = priority_df['priority_note'].where(
    priority_df['priority_note'] == '',
    priority_df['priority_note'] + ' / '
) + '전체 161개 상위 오류와 겹침'

priority_path = OUTPUT_DATA_DIR / 'ddri_rep15_station_error_priority_table.csv'
priority_df.to_csv(priority_path, index=False, encoding='utf-8-sig')

priority_df[['error_rank', 'station_id', 'station_name', 'station_group', 'cluster', 'mae', 'rmse', 'is_full_top20_overlap', 'priority_note']]


,error_rank,station_id,station_name,station_group,cluster,mae,rmse,is_full_top20_overlap,priority_note
0,1,2377,수서역 5번출구,아침 도착 업무 집중형,0,1.081768,1.573110,True,대표 15개 우선 해석 상위권 / 전체 161개 상위 오류와 겹침
1,2,2348,포스코사거리(기업은행),아침 도착 업무 집중형,1,0.949245,1.547188,True,대표 15개 우선 해석 상위권 / 전체 161개 상위 오류와 겹침
2,3,4917,일원에코파크 주차장,주거 도착형,2,0.791651,1.144001,False,대표 15개 우선 해석 상위권
3,4,2328,르네상스 호텔 사거리 역삼지하보도 7번출구 앞,업무/상업 혼합형,0,0.639666,0.920731,False,대표 15개 우선 해석 상위권
4,5,2359,"국립국악중,고교 정문 맞은편",외곽 주거형,4,0.649187,0.920162,False,대표 15개 우선 해석 상위권
5,6,4902,구역삼세무서 교차로,업무/상업 혼합형,0,0.630647,0.901145,False,
6,7,2320,도곡역 대치지구대 방향,생활권 혼합형,3,0.595651,0.818619,False,
7,8,3643,더시그넘하우스 앞,외곽 주거형,4,0.552798,0.794575,False,
8,9,2312,청담역 13번 출구 앞,주거 도착형,2,0.480443,0.689738,False,
9,10,2321,학여울역 사거리,생활권 혼합형,3,0.443784,0.641823,False,



## 2. 군집별 평균 오류 수준 확인

대표 15개는 군집별 3개 스테이션으로 구성되어 있으므로, 군집 그룹 단위 평균 RMSE를 보면 우선 해석할 군집이 어느 쪽인지 빠르게 읽을 수 있다.


In [3]:

cluster_summary = priority_df.groupby('station_group').agg(
    station_count=('station_id', 'count'),
    mean_rmse=('rmse', 'mean'),
    top_station_id=('station_id', 'first'),
    top_station_name=('station_name', 'first'),
).reset_index().sort_values('mean_rmse', ascending=False)

cluster_summary_path = OUTPUT_DATA_DIR / 'ddri_rep15_station_error_cluster_summary.csv'
cluster_summary.to_csv(cluster_summary_path, index=False, encoding='utf-8-sig')
cluster_summary


,station_group,station_count,mean_rmse,top_station_id,top_station_name
1,아침 도착 업무 집중형,3,1.214153,2377,수서역 5번출구
2,업무/상업 혼합형,3,0.777313,2328,르네상스 호텔 사거리 역삼지하보도 7번출구 앞
4,주거 도착형,3,0.742241,4917,일원에코파크 주차장
0,생활권 혼합형,3,0.680078,2320,도곡역 대치지구대 방향
3,외곽 주거형,3,0.663015,2359,"국립국악중,고교 정문 맞은편"



## 3. `2377`, `2348` 시간대별 오차 패턴 재생성

대표 15개 상위 오류 1, 2위이면서 전체 161개 상위 오류와도 겹치는 `2377`, `2348`를 대상으로, 대표 실험의 기본 모델(`LightGBM_RMSE(RMSE 기준 LightGBM)`)을 다시 학습해 시간대별 실제값-예측값 차이를 본다.

주의:

- 이 셀은 시간대 해석용 보조 분석이다.
- 기존 문서의 전체 점수표와 소수점 단위가 약간 다를 수 있으나, 목적은 `오차가 어느 시간대에 몰리는지`를 확인하는 것이다.


In [4]:

train_df = pd.read_csv(TRAIN_PATH, encoding='utf-8-sig')
test_df = pd.read_csv(TEST_PATH, encoding='utf-8-sig')

feature_cols = [
    'station_id', 'station_name', 'station_group', 'hour', 'cluster', 'mapped_dong_code',
    'weekday', 'month', 'holiday', 'temperature', 'humidity', 'precipitation', 'wind_speed'
]
cat_cols = ['station_id', 'station_name', 'station_group', 'cluster', 'mapped_dong_code', 'weekday', 'month', 'holiday', 'hour']

X_train = train_df[feature_cols].copy()
y_train = train_df['rental_count'].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df['rental_count'].copy()

for c in cat_cols:
    X_train[c] = X_train[c].astype('category')
    X_test[c] = X_test[c].astype('category')

lightgbm_rmse = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='regression',
)
lightgbm_rmse.fit(X_train, y_train, categorical_feature=cat_cols)

pred = lightgbm_rmse.predict(X_test)

test_eval = test_df[['station_id', 'station_name', 'station_group', 'date', 'hour', 'rental_count']].copy()
test_eval['prediction'] = pred
test_eval['residual'] = test_eval['rental_count'] - test_eval['prediction']
test_eval['abs_error'] = test_eval['residual'].abs()
test_eval['squared_error'] = test_eval['residual'] ** 2

target_ids = ['2377', '2348']
target_eval = test_eval[test_eval['station_id'].astype(str).isin(target_ids)].copy()

station_summary = target_eval.groupby(['station_id', 'station_name', 'station_group'], as_index=False).agg(
    actual_mean=('rental_count', 'mean'),
    predicted_mean=('prediction', 'mean'),
    mae=('abs_error', 'mean'),
    rmse=('squared_error', lambda s: (s.mean()) ** 0.5),
).sort_values('rmse', ascending=False)

hourly_patterns = target_eval.groupby(['station_id', 'station_name', 'station_group', 'hour'], as_index=False).agg(
    actual_mean=('rental_count', 'mean'),
    predicted_mean=('prediction', 'mean'),
    mae=('abs_error', 'mean'),
    rmse=('squared_error', lambda s: (s.mean()) ** 0.5),
    sample_count=('date', 'count'),
)
hourly_patterns['gap_mean'] = hourly_patterns['predicted_mean'] - hourly_patterns['actual_mean']

peak_hours = hourly_patterns.sort_values(['station_id', 'mae', 'rmse'], ascending=[True, False, False]).groupby('station_id').head(5)

hourly_path = OUTPUT_DATA_DIR / 'ddri_rep15_top2_station_hourly_error_patterns.csv'
summary_path = OUTPUT_DATA_DIR / 'ddri_rep15_top2_station_error_summary.csv'
peak_path = OUTPUT_DATA_DIR / 'ddri_rep15_top2_station_peak_error_hours.csv'

hourly_patterns.to_csv(hourly_path, index=False, encoding='utf-8-sig')
station_summary.to_csv(summary_path, index=False, encoding='utf-8-sig')
peak_hours.to_csv(peak_path, index=False, encoding='utf-8-sig')

station_summary


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002443 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 813
[LightGBM] [Info] Number of data points in the train set: 263160, number of used features: 13
[LightGBM] [Info] Start training from score 0.724514


,station_id,station_name,station_group,actual_mean,predicted_mean,mae,rmse
1,2377,수서역 5번출구,아침 도착 업무 집중형,1.822603,1.930406,1.184462,1.716840
0,2348,포스코사거리(기업은행),아침 도착 업무 집중형,1.475228,1.861062,1.053046,1.690407



## 4. 피크 오차 시간대 확인

각 스테이션별로 MAE(평균절대오차)가 큰 시간대 상위 5개를 보면, 어떤 시간대의 수요 구조를 다음 분석에서 더 봐야 할지 정할 수 있다.


In [5]:

peak_hours[['station_id', 'station_name', 'hour', 'actual_mean', 'predicted_mean', 'mae', 'rmse', 'gap_mean']]


,station_id,station_name,hour,actual_mean,predicted_mean,mae,rmse,gap_mean
18,2348,포스코사거리(기업은행),18,6.939726,8.006162,3.392415,4.769775,1.066436
17,2348,포스코사거리(기업은행),17,5.676712,7.495691,2.615563,3.560949,1.818978
16,2348,포스코사거리(기업은행),16,2.827397,3.539672,1.534984,1.977867,0.712275
19,2348,포스코사거리(기업은행),19,2.169863,2.680857,1.527397,1.932763,0.510994
13,2348,포스코사거리(기업은행),13,1.249315,2.132493,1.311348,1.600581,0.883178
42,2377,수서역 5번출구,18,5.852055,6.008468,2.602583,3.413188,0.156413
43,2377,수서역 5번출구,19,3.923288,3.875234,2.382748,3.158294,-0.048054
41,2377,수서역 5번출구,17,4.169863,4.602516,2.045808,2.633699,0.432653
44,2377,수서역 5번출구,20,2.657534,2.645472,1.673139,2.181969,-0.012062
40,2377,수서역 5번출구,16,2.969863,3.068608,1.643067,2.132021,0.098745



## 5. 현재 해석

- 대표 15개 기준 상위 오류 우선 해석 대상은 `2377`, `2348`, `4917`, `2328`, `2359`다.
- 이 중 `2377`, `2348`는 전체 161개 상위 오류 Top20(상위 20개)와도 겹치므로, 대표 15개와 전체 161개를 연결하는 핵심 사례로 볼 수 있다.
- `2377`, `2348` 모두 큰 오차가 주로 `16~19시` 구간에 몰려 있다.
- 따라서 다음 단계는 `2377`, `2348`의 날짜별 피크 오차 분포 확인 또는 대표 15개 상위 5개 전체의 시간대 패턴 확장이다.
